In [1]:
import pandas as pd
import numpy as np
import re
import time
from pathlib import Path
from rank_bm25 import BM25Okapi

DATA = Path(r"d:\ML Project - E-commerce product search with semantic ranking\data")

corpus = pd.read_parquet(DATA / "corpus.parquet")
queries = pd.read_parquet(DATA / "queries.parquet")
qrels = pd.read_parquet(DATA / "qrels.parquet")

print("Corpus:", len(corpus))
print("Queries:", len(queries))
print("Judgments:", len(qrels))

Corpus: 40063
Queries: 500
Judgments: 10125


In [2]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

tokenize("Nike Men's Air Max 270 Running Shoes, Black")

['nike', 'men', 's', 'air', 'max', '270', 'running', 'shoes', 'black']

In [3]:
start = time.time()

tokenized_corpus = [tokenize(t) for t in corpus["search_text"]]
bm25 = BM25Okapi(tokenized_corpus)

print(f"Index built in {time.time() - start:.1f} seconds")

Index built in 5.5 seconds


In [5]:
product_ids = corpus["product_id"].values

def search_bm25(query, top_k=10):
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(product_ids[i], float(scores[i])) for i in top_idx]

In [6]:
def show_results(query, top_k=5):
    print("QUERY:", query, "\n")
    for pid, score in search_bm25(query, top_k):
        title = corpus.loc[corpus["product_id"] == pid, "product_title"].iloc[0]
        print(f"{score:6.2f}  {title[:90]}")

show_results("running shoes for men")

QUERY: running shoes for men 

 19.51  Powerstep Pulse Performance Insole, Running Shoe Insert, Plantar Fasciitis and Neutral Arc
 18.79  Flats Shoes Women Running Shoes Men Cowgirl Boots for Women Mens Work Boots,Shoes for Wome
 17.90  JSPOYOU Men's Breathable Running Shoe Casual Shoes Sports Shoes Grey
 17.87  TSIODFO Sneakers for Men Sport Running Shoes Athletic Tennis Walking Shoes Fashion Jogging
 17.69  New Balance Men's Q Speed Run Crew Short, Black, XX-Large


In [7]:
show_results("something to keep my coffee hot")
show_results("sofa for small room")

QUERY: something to keep my coffee hot 

 22.40  12oz Travel Mug, MOMSIV Insulated Coffee Cup with Leakproof Lid, Vacuum Stainless Steel Do
 21.63  12oz Insulated Coffee Travel Mug Stainless Steel Vacuum Coffee Cup Leakproof with Screw Li
 19.10  Custom Variety Pack Coffee, Tea, Cappuccino and Hot Chocolate Variety Sampler Pack for Keu
 18.47  [200 Pack] 16 oz. Disposable White Paper Hot Coffee Cups
 17.71  Eeffoc Is Coffee Spelled Backwards, As I Dont Give Eeffoc Until I Had My Coffee - Funny Co
QUERY: sofa for small room 

 21.56  Navy Blue Sectional Sofa Sleeper Bed,JULYFOX 900 LB Heavy Duty 115 inch Velvet Sofa Futon 
 19.37  Sofa Sectional Sofa 2 Piece Modern Contemporary for Living Room Futon Sofa Bed Couches and
 17.69  FDW Sofa Sectional Sofa for Living Room Futon Sofa Modern Sofa Couches and Sofas Furniture
 17.53  Furologee Console Table with 2 Drawers, Industrial Sofa Table, Entryway Table with 3-Tier 
 17.16  Vonanda Faux Leather Sofa Couch, Mid-Century Handmade with 74 inc

In [8]:
q = queries.iloc[0]
print("QUERY:", q["query"], "\n")

print("BM25 results:")
results = search_bm25(q["query"], 5)
labels = qrels[qrels["query_id"] == q["query_id"]].set_index("product_id")["esci_label"]

for pid, score in results:
    title = corpus.loc[corpus["product_id"] == pid, "product_title"].iloc[0]
    label = labels.get(pid, "not judged")
    print(f"[{label:>10}] {score:6.2f}  {title[:80]}")

QUERY: inflatable dildo 

BM25 results:
[         E]  25.12  Trinity Vibes Inflatable Suction Cup Dildo
[         S]  16.81  10" Monster Silicone Realistic Dildo with Dual Density Anal Dildo Huge Suction C
[not judged]  13.11  Clitoral Sucking Vibrator,G Spot Vibrating Realistic Dildo 2 in 1 Vibrators Clit
[not judged]  13.03  Dildo Elastic D'ildo for Adullt Women's Body Stimulation and Relaxation Mǎssage 
[not judged]  12.33  UIBETR 10 Inch Silicone Didlo Realistic Flexible Black Long Massage Massag Big D


In [9]:
start = time.time()
for qtext in queries["query"].head(20):
    search_bm25(qtext, 10)
avg = (time.time() - start) / 20
print(f"Average time per query: {avg*1000:.0f} ms")

Average time per query: 97 ms


In [10]:
rows = []
for _, row in queries.iterrows():
    for rank, (pid, score) in enumerate(search_bm25(row["query"], 100), start=1):
        rows.append((row["query_id"], pid, rank, score))

bm25_run = pd.DataFrame(rows, columns=["query_id", "product_id", "rank", "score"])
bm25_run.to_parquet(DATA / "run_bm25.parquet", index=False)

print("Saved! Rows:", len(bm25_run))

Saved! Rows: 50000
